# Male Female Object Detection Training (Colab)

This notebook trains a YOLOv8 model using your Google Drive folder:
https://drive.google.com/drive/folders/1nTTQCZywDSh7wFlk_LzzUJZydkSogLO-?usp=sharing

It supports two dataset cases automatically:
1. Real YOLO detection labels (images + labels).
2. Classification-only folders (Training/male, Training/female, Validation/male, Validation/female).

If labels are missing, it creates pseudo detection labels with one full-image box per image.

In [2]:
import sys

!{sys.executable} -m pip install -q -U ultralytics gdown pyyaml

import ultralytics
print('ultralytics version:', ultralytics.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics version: 8.4.37


In [5]:
from pathlib import Path
from google.colab import drive
import re
import shutil
import subprocess


drive.mount('/content/drive')

DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1nTTQCZywDSh7wFlk_LzzUJZydkSogLO-?usp=sharing'
WORK_DIR = Path('/content/male_female_project')
DOWNLOAD_DIR = WORK_DIR / 'downloaded_drive_folder'
PREPARED_DIR = WORK_DIR / 'prepared_yolo'

# Optional: set this if your dataset is already visible in mounted Drive.
# Example: '/content/drive/MyDrive/malefemal'
LOCAL_DATASET_ROOT = None

WORK_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_DIR.mkdir(parents=True, exist_ok=True)


def extract_folder_id(url: str):
    match = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
    return match.group(1) if match else None


def choose_drive_shortcut_root(url: str):
    folder_id = extract_folder_id(url)
    if not folder_id:
        return None

    shortcut_root = Path('/content/drive/.shortcut-targets-by-id') / folder_id
    if not shortcut_root.exists() or not shortcut_root.is_dir():
        return None

    child_dirs = sorted([p for p in shortcut_root.iterdir() if p.is_dir()])
    if len(child_dirs) == 1:
        return child_dirs[0]
    return shortcut_root


def print_top_entries(path: Path, limit: int = 40):
    print(f'Top-level entries in {path}:')
    for item in sorted(path.iterdir())[:limit]:
        kind = 'DIR' if item.is_dir() else 'FILE'
        print(f'- [{kind}] {item.name}')


DATA_SOURCE_ROOT = None

if LOCAL_DATASET_ROOT:
    manual_root = Path(LOCAL_DATASET_ROOT)
    if manual_root.exists():
        DATA_SOURCE_ROOT = manual_root
        print('Using LOCAL_DATASET_ROOT:', DATA_SOURCE_ROOT)
    else:
        print('LOCAL_DATASET_ROOT does not exist, falling back to other methods.')

if DATA_SOURCE_ROOT is None:
    shortcut_root = choose_drive_shortcut_root(DRIVE_FOLDER_URL)
    if shortcut_root is not None:
        DATA_SOURCE_ROOT = shortcut_root
        print('Using mounted Drive shortcut path:', DATA_SOURCE_ROOT)

if DATA_SOURCE_ROOT is None:
    if DOWNLOAD_DIR.exists():
        shutil.rmtree(DOWNLOAD_DIR)
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    print('Downloading shared Drive folder with gdown...')
    cmd = ['gdown', '--folder', DRIVE_FOLDER_URL, '-O', str(DOWNLOAD_DIR)]
    result = subprocess.run(cmd, check=False, text=True, capture_output=True)

    if result.stdout:
        print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        print(f'gdown exited with code {result.returncode}. Continuing with any downloaded files.')

    downloaded_items = list(DOWNLOAD_DIR.glob('*'))
    if not downloaded_items:
        raise RuntimeError(
            'No files were downloaded and no mounted Drive shortcut path was found. '
            'Open the shared folder link, add it as a shortcut to MyDrive, and rerun this cell.'
        )

    DATA_SOURCE_ROOT = DOWNLOAD_DIR

print('Data source root:', DATA_SOURCE_ROOT)
if DATA_SOURCE_ROOT.exists():
    print_top_entries(DATA_SOURCE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using mounted Drive shortcut path: /content/drive/.shortcut-targets-by-id/1nTTQCZywDSh7wFlk_LzzUJZydkSogLO-/malefemal
Data source root: /content/drive/.shortcut-targets-by-id/1nTTQCZywDSh7wFlk_LzzUJZydkSogLO-/malefemal
Top-level entries in /content/drive/.shortcut-targets-by-id/1nTTQCZywDSh7wFlk_LzzUJZydkSogLO-/malefemal:
- [DIR] Training
- [DIR] Validation


In [1]:
import random
import shutil
import tarfile
import zipfile
from pathlib import Path
import yaml

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
TRAIN_KEYS = ('train', 'training')
VAL_KEYS = ('val', 'valid', 'validation', 'test')
VAL_RATIO = 0.2
SPLIT_SEED = 42


def norm_name(name: str) -> str:
    return ''.join(ch.lower() for ch in name if ch.isalnum())


def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTS


def list_images(folder: Path):
    if not folder.exists() or not folder.is_dir():
        return []
    return sorted([p for p in folder.iterdir() if is_image_file(p)])


def find_dir(parent: Path, names):
    if not parent.exists() or not parent.is_dir():
        return None
    name_set = set(names)
    for child in parent.iterdir():
        if child.is_dir() and child.name in name_set:
            return child
    return None


def find_dir_fuzzy(parent: Path, keywords):
    if not parent.exists() or not parent.is_dir():
        return None
    for child in parent.iterdir():
        if not child.is_dir():
            continue
        normalized = norm_name(child.name)
        if any(key in normalized for key in keywords):
            return child
    return None


def find_split_dirs(parent: Path):
    train_dir = find_dir(parent, ['train', 'training', 'Train', 'Training'])
    val_dir = find_dir(parent, ['val', 'valid', 'validation', 'Val', 'Valid', 'Validation', 'test', 'Test'])
    if train_dir is None:
        train_dir = find_dir_fuzzy(parent, TRAIN_KEYS)
    if val_dir is None:
        val_dir = find_dir_fuzzy(parent, VAL_KEYS)
    return train_dir, val_dir


def find_images_labels_dirs(parent: Path):
    images_dir = find_dir(parent, ['images', 'Images'])
    labels_dir = find_dir(parent, ['labels', 'Labels'])
    if images_dir is None:
        images_dir = find_dir_fuzzy(parent, ('image', 'images'))
    if labels_dir is None:
        labels_dir = find_dir_fuzzy(parent, ('label', 'labels', 'annotation', 'annotations'))
    return images_dir, labels_dir


def has_class_subdirs(folder: Path) -> bool:
    if folder is None or not folder.exists() or not folder.is_dir():
        return False
    return any(child.is_dir() for child in folder.iterdir())


def maybe_unpack_archives(download_root: Path):
    extracted_any = False
    extract_root = download_root / '_extracted'
    archive_files = []

    for path in download_root.rglob('*'):
        if not path.is_file():
            continue
        lower = path.name.lower()
        if lower.endswith('.zip') or lower.endswith('.tar') or lower.endswith('.tar.gz') or lower.endswith('.tgz'):
            archive_files.append(path)

    for archive in archive_files:
        target_dir = extract_root / archive.stem
        target_dir.mkdir(parents=True, exist_ok=True)
        try:
            if archive.name.lower().endswith('.zip'):
                with zipfile.ZipFile(archive, 'r') as zf:
                    zf.extractall(target_dir)
            else:
                with tarfile.open(archive, 'r:*') as tf:
                    tf.extractall(target_dir)
            extracted_any = True
            print(f'Extracted archive: {archive}')
        except Exception as exc:
            print(f'Skipped archive {archive}: {exc}')

    return extracted_any


def detect_dataset_root(scan_root: Path):
    candidates = [scan_root] + [p for p in scan_root.rglob('*') if p.is_dir()]
    for root in candidates:
        train_cls, val_cls = find_split_dirs(root)
        if train_cls and val_cls and has_class_subdirs(train_cls) and has_class_subdirs(val_cls):
            return root, 'classification'
        if train_cls and not val_cls and has_class_subdirs(train_cls):
            return root, 'classification_train_only'

        img_root, lbl_root = find_images_labels_dirs(root)
        if img_root and lbl_root:
            train_images, val_images = find_split_dirs(img_root)
            train_labels, val_labels = find_split_dirs(lbl_root)
            if train_images and val_images and train_labels and val_labels:
                return root, 'yolo_images_labels'
            if train_images and train_labels and not val_images and not val_labels:
                return root, 'yolo_images_labels_train_only'

        train_root, val_root = find_split_dirs(root)
        if train_root:
            train_images, train_labels = find_images_labels_dirs(train_root)
            if train_images and train_labels:
                if val_root:
                    val_images, val_labels = find_images_labels_dirs(val_root)
                    if val_images and val_labels:
                        return root, 'yolo_split_first'
                else:
                    return root, 'yolo_split_first_train_only'

    known_dirs = [str(p.relative_to(scan_root)) for p in sorted(scan_root.rglob('*')) if p.is_dir()]
    preview = '\n'.join(known_dirs[:80])
    raise FileNotFoundError(
        'Could not find a supported dataset structure inside the data source.\n'
        'First detected directories:\n'
        f'{preview}'
    )


def ensure_clean_prepared(prepared_root: Path):
    if prepared_root.exists():
        shutil.rmtree(prepared_root)
    for split in ('train', 'val'):
        (prepared_root / 'images' / split).mkdir(parents=True, exist_ok=True)
        (prepared_root / 'labels' / split).mkdir(parents=True, exist_ok=True)


def copy_yolo_split(src_images: Path, src_labels: Path, split: str, prepared_root: Path):
    copied = 0
    skipped_missing_label = 0
    for idx, img in enumerate(list_images(src_images), start=1):
        label = src_labels / f'{img.stem}.txt'
        if not label.exists():
            skipped_missing_label += 1
            continue
        out_stem = f'{split}_{idx:06d}'
        shutil.copy2(img, prepared_root / 'images' / split / f'{out_stem}{img.suffix.lower()}')
        shutil.copy2(label, prepared_root / 'labels' / split / f'{out_stem}.txt')
        copied += 1
    return copied, skipped_missing_label


def copy_yolo_train_only_with_auto_val(src_images: Path, src_labels: Path, prepared_root: Path, val_ratio=0.2, seed=42):
    pairs = []
    skipped_missing_label = 0

    for img in list_images(src_images):
        label = src_labels / f'{img.stem}.txt'
        if not label.exists():
            skipped_missing_label += 1
            continue
        pairs.append((img, label))

    if not pairs:
        return {'train': 0, 'val': 0}, skipped_missing_label

    rng = random.Random(seed)
    rng.shuffle(pairs)

    n_val = int(len(pairs) * val_ratio)
    if len(pairs) >= 2:
        n_val = max(1, n_val)
    n_val = min(n_val, len(pairs) - 1) if len(pairs) > 1 else 0

    val_pairs = pairs[:n_val]
    train_pairs = pairs[n_val:]

    for idx, (img, label) in enumerate(train_pairs, start=1):
        out_stem = f'train_{idx:06d}'
        shutil.copy2(img, prepared_root / 'images' / 'train' / f'{out_stem}{img.suffix.lower()}')
        shutil.copy2(label, prepared_root / 'labels' / 'train' / f'{out_stem}.txt')

    for idx, (img, label) in enumerate(val_pairs, start=1):
        out_stem = f'val_{idx:06d}'
        shutil.copy2(img, prepared_root / 'images' / 'val' / f'{out_stem}{img.suffix.lower()}')
        shutil.copy2(label, prepared_root / 'labels' / 'val' / f'{out_stem}.txt')

    return {'train': len(train_pairs), 'val': len(val_pairs)}, skipped_missing_label


def collect_class_names(train_root: Path):
    names = sorted([d.name for d in train_root.iterdir() if d.is_dir()])
    if not names:
        raise RuntimeError('No class folders found in Training directory.')
    return names


def split_items_for_val(items, val_ratio=0.2, seed=42):
    shuffled = list(items)
    rng = random.Random(seed)
    rng.shuffle(shuffled)

    n_val = int(len(shuffled) * val_ratio)
    if len(shuffled) >= 2:
        n_val = max(1, n_val)
    n_val = min(n_val, len(shuffled) - 1) if len(shuffled) > 1 else 0

    return shuffled[n_val:], shuffled[:n_val]


def write_classification_detection_labels(images, class_id: int, split: str, class_name: str, prepared_root: Path, start_index: int):
    count = 0
    current = start_index
    for img in images:
        current += 1
        out_stem = f'{class_name}_{current:07d}'
        out_img = prepared_root / 'images' / split / f'{out_stem}{img.suffix.lower()}'
        out_lbl = prepared_root / 'labels' / split / f'{out_stem}.txt'
        shutil.copy2(img, out_img)
        out_lbl.write_text(f"{class_id} 0.5 0.5 1.0 1.0\n", encoding='utf-8')
        count += 1
    return count, current


def prepare_from_classification(dataset_root: Path, prepared_root: Path, val_ratio=0.2, seed=42):
    train_root, val_root = find_split_dirs(dataset_root)
    if train_root is None:
        raise RuntimeError('Could not locate a Training split folder for classification dataset.')

    class_names = collect_class_names(train_root)
    if len(class_names) < 2:
        raise RuntimeError(
            f'Only one class folder was found: {class_names}. '
            'This usually means the Drive download was partial. '
            'Add the shared folder as a shortcut to MyDrive and rerun Cell 3.'
        )

    class_to_id = {name: idx for idx, name in enumerate(class_names)}
    counts = {'train': 0, 'val': 0}

    if val_root is None:
        print('Validation folder not found. Building validation split from Training data.')

    for class_name in class_names:
        train_class_dir = train_root / class_name
        if not train_class_dir.exists():
            continue

        train_imgs = list_images(train_class_dir)
        val_imgs = []

        if val_root is not None:
            val_class_dir = val_root / class_name
            if val_class_dir.exists():
                val_imgs = list_images(val_class_dir)
        else:
            train_imgs, val_imgs = split_items_for_val(train_imgs, val_ratio=val_ratio, seed=seed + class_to_id[class_name])

        added, _ = write_classification_detection_labels(
            train_imgs,
            class_to_id[class_name],
            'train',
            class_name,
            prepared_root,
            counts['train']
        )
        counts['train'] += added

        added, _ = write_classification_detection_labels(
            val_imgs,
            class_to_id[class_name],
            'val',
            class_name,
            prepared_root,
            counts['val']
        )
        counts['val'] += added

    print('WARNING: No bounding-box labels detected. Generated pseudo labels with full-image boxes.')
    return class_names, counts


def load_names_from_existing_yaml(dataset_root: Path):
    yaml_candidates = list(dataset_root.glob('*.yaml')) + list(dataset_root.glob('*.yml'))
    for yaml_path in yaml_candidates:
        try:
            data = yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
            names = data.get('names')
            if isinstance(names, list) and names:
                return [str(x) for x in names]
            if isinstance(names, dict) and names:
                try:
                    keys = sorted(names.keys(), key=lambda x: int(x))
                except Exception:
                    keys = sorted(names.keys())
                return [str(names[k]) for k in keys]
        except Exception:
            pass
    return None


def collect_class_ids_from_labels(labels_root: Path):
    class_ids = set()
    for lbl in labels_root.rglob('*.txt'):
        for line in lbl.read_text(encoding='utf-8', errors='ignore').splitlines():
            parts = line.strip().split()
            if len(parts) >= 5 and parts[0].isdigit():
                class_ids.add(int(parts[0]))
    return class_ids


def infer_names_from_labels(prepared_root: Path):
    class_ids = collect_class_ids_from_labels(prepared_root / 'labels')
    if not class_ids:
        raise RuntimeError('No valid YOLO labels found after preparation.')
    max_id = max(class_ids)
    return [f'class_{idx}' for idx in range(max_id + 1)]


SCAN_ROOT = Path(DATA_SOURCE_ROOT) if 'DATA_SOURCE_ROOT' in globals() else DOWNLOAD_DIR
if not SCAN_ROOT.exists():
    raise FileNotFoundError(f'Data source root does not exist: {SCAN_ROOT}')

_ = maybe_unpack_archives(SCAN_ROOT)
dataset_root, mode = detect_dataset_root(SCAN_ROOT)
print('Detected dataset root:', dataset_root)
print('Detected format:', mode)

ensure_clean_prepared(PREPARED_DIR)

if mode in ('classification', 'classification_train_only'):
    class_names, counts = prepare_from_classification(dataset_root, PREPARED_DIR, val_ratio=VAL_RATIO, seed=SPLIT_SEED)
else:
    if mode == 'yolo_images_labels':
        images_root, labels_root = find_images_labels_dirs(dataset_root)
        train_images, val_images = find_split_dirs(images_root)
        train_labels, val_labels = find_split_dirs(labels_root)

        train_copied, train_skipped = copy_yolo_split(train_images, train_labels, 'train', PREPARED_DIR)
        val_copied, val_skipped = copy_yolo_split(val_images, val_labels, 'val', PREPARED_DIR)
        counts = {'train': train_copied, 'val': val_copied}
        skipped_info = train_skipped + val_skipped

    elif mode == 'yolo_split_first':
        train_root, val_root = find_split_dirs(dataset_root)
        train_images, train_labels = find_images_labels_dirs(train_root)
        val_images, val_labels = find_images_labels_dirs(val_root)

        train_copied, train_skipped = copy_yolo_split(train_images, train_labels, 'train', PREPARED_DIR)
        val_copied, val_skipped = copy_yolo_split(val_images, val_labels, 'val', PREPARED_DIR)
        counts = {'train': train_copied, 'val': val_copied}
        skipped_info = train_skipped + val_skipped

    elif mode == 'yolo_images_labels_train_only':
        images_root, labels_root = find_images_labels_dirs(dataset_root)
        train_images, _ = find_split_dirs(images_root)
        train_labels, _ = find_split_dirs(labels_root)

        counts, skipped_info = copy_yolo_train_only_with_auto_val(
            train_images,
            train_labels,
            PREPARED_DIR,
            val_ratio=VAL_RATIO,
            seed=SPLIT_SEED
        )
        print('Validation split was auto-generated from training data.')

    else:
        train_root, _ = find_split_dirs(dataset_root)
        train_images, train_labels = find_images_labels_dirs(train_root)

        counts, skipped_info = copy_yolo_train_only_with_auto_val(
            train_images,
            train_labels,
            PREPARED_DIR,
            val_ratio=VAL_RATIO,
            seed=SPLIT_SEED
        )
        print('Validation split was auto-generated from training data.')

    print(f'Skipped images without labels: {skipped_info}')

    class_names = load_names_from_existing_yaml(dataset_root)
    if class_names is None:
        class_names = infer_names_from_labels(PREPARED_DIR)

    class_ids = collect_class_ids_from_labels(PREPARED_DIR / 'labels')
    if class_ids and max(class_ids) >= len(class_names):
        print('Class IDs exceed provided class names. Falling back to inferred placeholder names.')
        class_names = infer_names_from_labels(PREPARED_DIR)

if counts['train'] == 0 or counts['val'] == 0:
    raise RuntimeError(
        f'Prepared dataset is empty or missing split. Counts: {counts}. '
        'If this is from gdown, rerun Cell 3 after adding the shared folder as a MyDrive shortcut.'
    )

DATA_YAML_PATH = PREPARED_DIR / 'dataset.yaml'
yaml_data = {
    'path': str(PREPARED_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(class_names),
    'names': class_names,
}
DATA_YAML_PATH.write_text(yaml.safe_dump(yaml_data, sort_keys=False), encoding='utf-8')

print('Prepared dataset at:', PREPARED_DIR)
print('Counts:', counts)
print('Classes:', class_names)
print('YAML path:', DATA_YAML_PATH)
print(DATA_YAML_PATH.read_text())

NameError: name 'DOWNLOAD_DIR' is not defined

In [ ]:
import os
import subprocess
import torch

if not torch.cuda.is_available():
    raise EnvironmentError(
        'GPU is not enabled. In Colab: Runtime > Change runtime type > Hardware accelerator = GPU, then rerun.'
    )

print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0))
print('CUDA version:', torch.version.cuda)
print('PyTorch:', torch.__version__)

torch.backends.cudnn.benchmark = True
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

_ = subprocess.run(['nvidia-smi'])

In [ ]:
import os
import time
from pathlib import Path

import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise EnvironmentError(
        'GPU not detected. In Colab: Runtime > Change runtime type > Hardware accelerator = GPU, then reconnect and rerun.'
    )

BASE_WEIGHTS = 'yolov8n.pt'  # Fastest option for quick epochs. Use yolov8s.pt for potentially better accuracy.
EPOCHS = 40
IMGSZ = 640
BATCH = -1  # Auto-batch uses GPU memory efficiently for higher utilization.
WORKERS = min(8, os.cpu_count() or 2)
PROJECT = '/content/runs/detect'
RUN_NAME = f'male_female_od_{int(time.time())}'
CACHE_MODE = 'ram'

model = YOLO(BASE_WEIGHTS)

train_kwargs = dict(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=PROJECT,
    name=RUN_NAME,
    cache=CACHE_MODE,
    workers=WORKERS,
    amp=True,
    cos_lr=True,
    close_mosaic=10,
    optimizer='auto',
    plots=True,
    exist_ok=True,
    verbose=True,
    deterministic=False
    # deterministic=False allows faster cudnn kernels.
    
)

try:
    train_results = model.train(**train_kwargs)
except RuntimeError as exc:
    if 'out of memory' in str(exc).lower():
        print('GPU OOM detected. Retrying with safer settings...')
        torch.cuda.empty_cache()
        train_kwargs.update({
            'batch': 16,
            'cache': False,
            'workers': max(2, min(6, WORKERS)),
        })
        train_results = model.train(**train_kwargs)
    else:
        raise

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'
LAST_WEIGHTS = RUN_DIR / 'weights' / 'last.pt'

print('Training finished.')
print('Run directory:', RUN_DIR)
print('Best weights:', BEST_WEIGHTS, BEST_WEIGHTS.exists())
print('Last weights:', LAST_WEIGHTS, LAST_WEIGHTS.exists())

NameError: name 'DATA_YAML_PATH' is not defined

In [ ]:
from pathlib import Path
from ultralytics import YOLO

run_dir = RUN_DIR if 'RUN_DIR' in globals() else (Path(PROJECT) / RUN_NAME)
best_weights = BEST_WEIGHTS if 'BEST_WEIGHTS' in globals() else (run_dir / 'weights' / 'best.pt')
last_weights = LAST_WEIGHTS if 'LAST_WEIGHTS' in globals() else (run_dir / 'weights' / 'last.pt')

weights_for_eval = best_weights if best_weights.exists() else last_weights
if not weights_for_eval.exists():
    raise FileNotFoundError(f'No trained weights found. Checked: {best_weights} and {last_weights}')

print('Evaluating with:', weights_for_eval)
best_model = YOLO(str(weights_for_eval))
metrics = best_model.val(data=str(DATA_YAML_PATH), imgsz=IMGSZ, device=0)
print(metrics)

val_images = sorted((PREPARED_DIR / 'images' / 'val').glob('*'))
sample_sources = [str(p) for p in val_images[:10]]
if sample_sources:
    _ = best_model.predict(
        source=sample_sources,
        conf=0.25,
        save=True,
        project='/content/runs/predict',
        name=RUN_NAME,
        exist_ok=True,
        device=0
    )
    print('Saved prediction images to /content/runs/predict/' + RUN_NAME)
else:
    print('No validation images found for preview predictions.')

In [ ]:
import shutil
import zipfile
from datetime import datetime
from pathlib import Path
from google.colab import files

EXPORT_TO_DRIVE = True
DOWNLOAD_WEIGHTS_ZIP = True

run_dir = RUN_DIR if 'RUN_DIR' in globals() else (Path(PROJECT) / RUN_NAME)
if not run_dir.exists():
    raise FileNotFoundError(f'Training output not found: {run_dir}')

best_src = BEST_WEIGHTS if 'BEST_WEIGHTS' in globals() else (run_dir / 'weights' / 'best.pt')
last_src = LAST_WEIGHTS if 'LAST_WEIGHTS' in globals() else (run_dir / 'weights' / 'last.pt')

if EXPORT_TO_DRIVE:
    export_root = Path('/content/drive/MyDrive/male_female_training_runs')
    export_root.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    dst_dir = export_root / f'{RUN_NAME}_{timestamp}'
    shutil.copytree(run_dir, dst_dir)
    print('Exported run to Drive:', dst_dir)
else:
    print('Drive export skipped.')

if DOWNLOAD_WEIGHTS_ZIP:
    if not best_src.exists() and not last_src.exists():
        raise FileNotFoundError(f'No weights to download. Checked: {best_src} and {last_src}')

    zip_path = Path('/content') / f'{RUN_NAME}_weights.zip'
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        if best_src.exists():
            zf.write(best_src, arcname='best.pt')
        if last_src.exists():
            zf.write(last_src, arcname='last.pt')

        results_csv = run_dir / 'results.csv'
        if results_csv.exists():
            zf.write(results_csv, arcname='results.csv')

        if 'DATA_YAML_PATH' in globals() and Path(DATA_YAML_PATH).exists():
            zf.write(Path(DATA_YAML_PATH), arcname='dataset.yaml')

    print('Downloading weights package:', zip_path)
    files.download(str(zip_path))
else:
    print('Download skipped.')